# **Demo: Chain of Thought (CoT) Prompting with Hugging Face Models**

### **Steps to Follow:**
- **Step 1:** Set up Hugging Face Authentication and Model Pipeline
- **Step 2:** Define a Function to Get Completion
- **Step 3:** Evaluate Standard Direct Prompting (Baseline Without CoT)
- **Step 4:** Implement Zero-Shot Chain-of-Thought ("Let's think step by step")
- **Step 5:** Implement Few-Shot Chain-of-Thought for Complex Engineering Logic

### **Step 1: Set up Hugging Face Authentication and Model Pipeline**
- Installs required libraries (`transformers`, `torch`, `accelerate`).
- Securely loads your Hugging Face access token from Google Colab Secrets (`HF_TOKEN`).
- Loads `Qwen/Qwen2.5-0.5B-Instruct` onto the GPU with half precision (`torch.float16`).

In [1]:
# Step 1: Install dependencies and load Llama-3.2-3B-Instruct
!pip install -q transformers torch accelerate

import torch
from google.colab import userdata
from transformers import pipeline

# Retrieve token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Initialize text-generation pipeline on GPU
generator = pipeline(
    "text-generation",
    #model="meta-llama/Llama-3.2-3B-Instruct",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    token=hf_token,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ Qwen2.5-0.5B-Instruct initialized successfully on GPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen2.5-0.5B-Instruct initialized successfully on GPU.


### **Step 2: Define a Function to Get Completion**
- Wraps generation calls in a reusable helper function.
- Allocates sufficient `max_new_tokens` (200+) so the model has token space to generate intermediate reasoning steps before delivering the conclusion.
- Uses greedy decoding (`do_sample=False`) for reproducible, deterministic reasoning traces.

In [2]:
# Step 2: Reusable completion helper
def get_completion(prompt, max_tokens=250):
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=max_tokens, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

print("✅ Completion helper ready.")

✅ Completion helper ready.


### **Step 3: Evaluate Standard Direct Prompting (Baseline Without CoT)**
- Direct prompting asks the model for an immediate final answer without reasoning steps.
- In complex engineering capacity and latency calculations, small-to-medium models often fail or hallucinate when forced to output a final number directly.

In [8]:
# Baseline task: Server capacity planning
direct_prompt = """
A microservice cluster processes 1,800 requests per minute.
Each server instance can reliably handle 15 requests per second.
We need to maintain a 20% capacity headroom for traffic spikes.
How many server instances must be provisioned?

Answer directly with just the final number.
"""

print("--- Baseline: Direct Prompting (No CoT) ---")
print(get_completion(direct_prompt, max_tokens=1000))

--- Baseline: Direct Prompting (No CoT) ---
To determine the number of server instances required to meet the 20% capacity headroom for traffic spikes while processing 1,800 requests per minute, we can follow these steps:

1. **Calculate the total number of requests processed in one hour:**
   \[
   \text{Requests per hour} = 1,800 \text{ requests/minute} \times 60 \text{ minutes/hour} = 108,000 \text{ requests/hour}
   \]

2. **Determine the number of requests that can be handled by each server instance in one hour:**
   \[
   \text{Requests per hour per server instance} = \frac{108,000 \text{ requests}}{15 \text{ requests/second}} = 7200 \text{ requests/s}
   \]

3. **Calculate the minimum number of server instances needed to process at least 7200 requests per hour:**
   Since each server instance can handle up to 15 requests per second, and there are 60 seconds in an hour:
   \[
   \text{Minimum server instances} = \left\lceil \frac{7200 \text{ requests/s}}{15 \text{ requests/s/serve

### **Step 4: Implement Zero-Shot Chain-of-Thought ("Let's think step by step")**
- Appending the reasoning trigger **"Think step-by-step before providing the final answer"** instructs the model to generate intermediate calculations and state transformations.
- This substantially increases mathematical and logical accuracy without requiring handcrafted examples.

In [9]:
# Zero-Shot CoT with the step-by-step reasoning directive
zero_shot_cot_prompt = """
A microservice cluster processes 1,800 requests per minute.
Each server instance can reliably handle 15 requests per second.
We need to maintain a 20% capacity headroom for traffic spikes.
How many server instances must be provisioned?

Let's think step by step. Show your calculations clearly, then state the final count of instances.
"""

print("--- Zero-Shot Chain of Thought ---")
print(get_completion(zero_shot_cot_prompt, max_tokens=1000))

--- Zero-Shot Chain of Thought ---
To determine the number of server instances required to ensure that the microservice cluster maintains a 20% capacity headroom for traffic spikes while processing 1,800 requests per minute, we'll follow these steps:

### Step 1: Calculate Total Requests Per Second
First, let's find out how many requests the cluster can process in one second.

- **Requests per minute:** 1,800 requests
- **Number of seconds in a minute:** 60 seconds

\[ \text{Requests per second} = \frac{\text{Requests per minute}}{60} = \frac{1,800}{60} = 30 \text{ requests/second} \]

### Step 2: Determine the Capacity Headroom
Next, we need to calculate the total capacity headroom needed to handle 20% more than the current capacity.

- **Current capacity:** 1,800 requests per minute (or 30 requests per second)
- **Headroom percentage:** 20%

\[ \text{Headroom} = \text{Current capacity} \times \left(1 + \frac{\text{Headroom percentage}}{100}\right) = 30 \times \left(1 + \frac{20}{100}

### **Step 5: Implement Few-Shot Chain-of-Thought for Complex Engineering Logic**
- **Few-Shot CoT** provides solved reasoning traces showing the model exactly *how* to break down algorithmic runtime complexity and memory analysis.
- It establishes a strict `Reasoning Trace:` and `Conclusion:` schema for consistent automated parsing.

In [12]:
# Few-Shot CoT prompt showing step-by-step architectural evaluation
few_shot_cot_prompt = """
Analyze the time and space complexity of the given algorithm.

Example 1:
Algorithm:
def find_duplicates(arr):
    seen = set()
    duplicates = []
    for x in arr:
        if x in seen:
            duplicates.append(x)
        seen.add(x)
    return duplicates

Reasoning Trace:
1. We iterate over `arr` of size N once, performing O(1) hash set lookups and insertions per element.
2. Total iteration time is O(N).
3. The `seen` set stores at most N elements, and `duplicates` stores at most N elements.
4. Total extra memory allocated scales linearly with input size.
Conclusion: Time Complexity: O(N) | Space Complexity: O(N)

---
Current Task:
Algorithm:
def matrix_search(matrix, target):
    # matrix is N x N
    for row in matrix:
        for val in row:
            if val == target:
                return True
    return False

Reasoning Trace:
"""

print("--- Few-Shot Chain of Thought (Complexity Analysis) ---")
print(get_completion(few_shot_cot_prompt, max_tokens=500))

--- Few-Shot Chain of Thought (Complexity Analysis) ---
1. We iterate over each row of the matrix using nested loops, which takes O(N^2) time.
2. Each inner loop iterates over all elements in the current row, resulting in O(N^2) iterations.
3. Therefore, the total number of iterations is O(N^2).
4. Since we have to check every element in the matrix, the space required is proportional to the size of the matrix.
5. Hence, the space complexity is O(N^2).

Conclusion: Time Complexity: O(N^2) | Space Complexity: O(N^2)
